In [ ]:
!pip install fastapi uvicorn nest-asyncio pyngrok
!pip install faiss-cpu sentence-transformers python-jose sqlalchemy

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

engine = create_engine("sqlite:///./test.*db")
SessionLocal = sessionmaker(bind=engine)
Base = declarative_base()

class User(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True)
    username = Column(String)
    password = Column(String)
    role = Column(String)

class Document(Base):
    __tablename__ = "documents"
    id = Column(Integer, primary_key=True)
    title = Column(String)
    company_name = Column(String)
    document_type = Column(String)
    content = Column(String)

Base.metadata.create_all(bind=engine)

In [ ]:
from jose import jwt
from datetime import datetime, timedelta

SECRET_KEY = "*****"

def create_token(user_id):
    payload = {
        "user_id": user_id,
        "exp": datetime.utcnow() + timedelta(hours=2)
    }
    return jwt.encode(payload, SECRET_KEY, algorithm="HS256")

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

index = faiss.IndexFlatL2(384)
chunks_store = []

def chunk_text(text):
    return text.split(".")

def add_document_rag(text):
    chunks = chunk_text(text)
    for chunk in chunks:
        emb = model.encode(chunk)
        index.add(np.array([emb]).astype("float32"))
        chunks_store.append(chunk)

def search_rag(query):
    emb = model.encode(query)
    D, I = index.search(np.array([emb]).astype("float32"), 5)
    return [chunks_store[i] for i in I[0]]

In [ ]:
from fastapi import FastAPI

app = FastAPI()

# ---------- AUTH ----------

@app.post("/auth/register")
def register(username: str, password: str, role: str):
    db = SessionLocal()
    user = User(username=username, password=password, role=role)
    db.add(user)
    db.commit()
    return {"msg": "User registered"}

@app.post("/auth/login")
def login(username: str, password: str):
    db = SessionLocal()
    user = db.query(User).filter(User.username==username).first()
    if user and user.password == password:
        token = create_token(user.id)
        return {"token": token}
    return {"error": "Invalid credentials"}

# ---------- DOCUMENT APIs ----------

@app.post("/documents/upload")
def upload(title: str, company: str, doc_type: str, content: str):
    db = SessionLocal()
    doc = Document(title=title, company_name=company, document_type=doc_type, content=content)
    db.add(doc)
    db.commit()

    add_document_rag(content)

    return {"msg": "Document stored"}

@app.get("/documents")
def get_all():
    db = SessionLocal()
    docs = db.query(Document).all()
    return docs

@app.get("/documents/{doc_id}")
def get_doc(doc_id: int):
    db = SessionLocal()
    doc = db.query(Document).filter(Document.id==doc_id).first()
    return doc

@app.delete("/documents/{doc_id}")
def delete_doc(doc_id: int):
    db = SessionLocal()
    doc = db.query(Document).filter(Document.id==doc_id).first()
    db.delete(doc)
    db.commit()
    return {"msg": "Deleted"}

@app.get("/documents/search")
def search_metadata(company: str):
    db = SessionLocal()
    docs = db.query(Document).filter(Document.company_name==company).all()
    return docs

# ---------- RAG ----------

@app.post("/rag/index-document")
def index_doc(text: str):
    add_document_rag(text)
    return {"msg": "Indexed"}

@app.post("/rag/search")
def rag_search(query: str):
    return {"results": search_rag(query)}

@app.get("/rag/context/{doc_id}")
def get_context(doc_id: int):
    db = SessionLocal()
    doc = db.query(Document).filter(Document.id==doc_id).first()
    return {"content": doc.content}

In [ ]:
import uvicorn

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

await server.serve()

In [ ]:
from pyngrok import ngrok

# 🔑 Your actual token (already added)
ngrok.set_auth_token("3****")

# 🔗 Create public tunnel
public_url = ngrok.connect(addr=8000, bind_tls=True)

print("Public URL:", public_url)